# 5장. 분석을 믿을 수 있게 만드는 데이터 전처리

이 노트북은 `book/chapters/ch05_data_preprocessing.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 데이터를 무조건 깨끗하게 만드는 것이 아니라, 분석 결과를 신뢰할 수 있도록 데이터의 상태를 확인하고 처리 기준을 남기는 것입니다.


## 0. 제출 정보
- 이름: 유은송
- GitHub ID: song-03
- 작성일: 2026.09.23. 
- 최종 제출 URL: https://github.com/song-03/llm-data-analysis-study/blob/main/chapter05/chapter05.ipynb

##### 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 원본 데이터는 `data/raw/`에 그대로 둡니다.
- 전처리 결과는 `data/processed/`에 별도로 저장합니다.
- 전처리 요약 보고서는 `reports/ch05_preprocessing_summary.md`에 저장합니다.
- 코드가 실행되었다고 끝이 아니라, 처리 기준을 설명할 수 있어야 합니다.


##### 1. 왜 전처리가 필요한가

현실의 데이터는 처음부터 분석하기 좋은 형태로 주어지지 않습니다.

예를 들어 다음과 같은 문제가 있을 수 있습니다.

| 문제 유형 | 예시 | 확인할 질문 |
|---|---|---|
| 결측치 | 나이가 비어 있는 고객 | 비어 있는 이유가 무엇인가? 삭제해도 되는가? |
| 중복 | 같은 고객 ID가 여러 번 등장 | 자연스러운 중복인가, 데이터 오류인가? |
| 타입 오류 | 가격이 문자열로 저장됨 | 계산 가능한 숫자형으로 바꿀 수 있는가? |
| 날짜 오류 | 주문일이 문자열로 저장됨 | 월별·요일별 분석에 사용할 수 있는가? |
| 문자열 표기 차이 | `Seoul`, ` Seoul`, `SEOUL` | 같은 값을 하나의 표기로 통일해야 하는가? |
| 이상값 | 수량이 음수이거나 가격이 0원 | 실제 의미가 있는 값인가, 입력 오류인가? |
| 파생 컬럼 필요 | 수량과 단가만 있고 주문금액이 없음 | 분석에 필요한 새 컬럼을 만들 수 있는가? |


##### 2. 원본 데이터와 전처리 데이터 분리하기

전처리에서 가장 중요한 원칙은 원본 데이터를 직접 덮어쓰지 않는 것입니다.

```text
data/
├─ raw/
│  ├─ customers.csv
│  ├─ products.csv
│  ├─ orders.csv
│  └─ order_items.csv
└─ processed/
   ├─ customers_clean.csv
   ├─ products_clean.csv
   ├─ orders_clean.csv
   └─ order_items_clean.csv
```

원본과 전처리 결과를 분리하면 실수했을 때 원본으로 돌아갈 수 있고, 전처리 전후 차이를 비교할 수 있습니다.


##### 3. 패키지와 경로 설정

노트북이 `notebooks/` 폴더 안에서 실행되는 경우와 프로젝트 루트에서 실행되는 경우를 모두 고려해 경로를 설정합니다.


In [108]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == 'notebooks':
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('현재 실행 위치:', CURRENT_DIR)
print('프로젝트 루트:', PROJECT_ROOT)
print('원본 데이터 폴더:', RAW_DIR)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)


현재 실행 위치: c:\dev\llm-data-analysis-course\notebooks
프로젝트 루트: c:\dev\llm-data-analysis-course
원본 데이터 폴더: c:\dev\llm-data-analysis-course\data\raw
전처리 데이터 폴더: c:\dev\llm-data-analysis-course\data\processed
보고서 폴더: c:\dev\llm-data-analysis-course\reports


##### 4. 원본 데이터 불러오기

이번 장에서는 온라인 쇼핑몰 예제 데이터 4개를 사용합니다.

- `customers.csv`: 고객 정보
- `products.csv`: 상품 정보
- `orders.csv`: 주문 정보
- `order_items.csv`: 주문 상세 정보


In [109]:
customers = pd.read_csv(RAW_DIR / 'customers.csv')
products = pd.read_csv(RAW_DIR / 'products.csv')
orders = pd.read_csv(RAW_DIR / 'orders.csv')
order_items = pd.read_csv(RAW_DIR / 'order_items.csv')

print('원본 데이터 불러오기 완료')


원본 데이터 불러오기 완료


##### 5. 전처리 전 데이터 크기 기록하기

전처리 전 데이터 크기를 기록해 두면 나중에 어떤 데이터가 얼마나 바뀌었는지 비교할 수 있습니다.


In [110]:
raw_data = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

raw_shapes = pd.DataFrame({
    'dataset': list(raw_data.keys()),
    'rows': [df.shape[0] for df in raw_data.values()],
    'columns': [df.shape[1] for df in raw_data.values()],
})

raw_shapes


,dataset,rows,columns
0,customers,150,6
1,products,100,4
2,orders,300,5
3,order_items,764,5


In [111]:
for name, df in raw_data.items():
    print(f'[{name}]')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print()


[customers]
shape: (150, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
shape: (100, 4)
columns: ['product_id', 'product_name', 'category', 'price']

[orders]
shape: (300, 5)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']

[order_items]
shape: (764, 5)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']



##### 6. 원본을 복사해서 전처리 시작하기

전처리 과정에서는 원본 DataFrame을 직접 수정하지 않고 복사본을 사용합니다.


In [112]:
customers_clean = customers.copy()
products_clean = products.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()

clean_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

print('전처리용 복사본 생성 완료')


전처리용 복사본 생성 완료


##### 7. 결측치 확인하기

결측치를 처리하기 전에 먼저 어디에 얼마나 비어 있는 값이 있는지 확인합니다. 개수와 비율을 함께 보면 데이터 크기에 따른 차이를 이해하기 쉽습니다.


In [113]:
for name, df in clean_data.items():
    print(f'\n[{name}] 결측치 개수')
    print(df.isna().sum())



[customers] 결측치 개수
customer_id    0
name           0
gender         0
age            0
city           0
signup_date    0
dtype: int64

[products] 결측치 개수
product_id      0
product_name    0
category        0
price           0
dtype: int64

[orders] 결측치 개수
order_id          0
customer_id       0
order_date        0
payment_method    0
order_status      0
dtype: int64

[order_items] 결측치 개수
order_item_id    0
order_id         0
product_id       0
quantity         0
unit_price       0
dtype: int64


In [114]:
def missing_summary(df):
    summary = pd.DataFrame({
        'missing_count': df.isna().sum(),
        'missing_ratio': (df.isna().mean() * 100).round(2),
    })
    return summary.sort_values('missing_count', ascending=False)

missing_summary(customers_clean)


,missing_count,missing_ratio
customer_id,0,0.0
name,0,0.0
gender,0,0.0
age,0,0.0
city,0,0.0
signup_date,0,0.0


##### 8. 결측치 처리하기

결측치 처리는 컬럼의 의미에 따라 달라집니다.

- 나이처럼 숫자형 컬럼은 중앙값으로 대체할 수 있습니다.
- 도시처럼 범주형 컬럼은 `Unknown`으로 남겨 분석에서 구분할 수 있습니다.

실제 업무에서는 결측치가 왜 생겼는지 먼저 확인해야 합니다.


In [115]:
if 'age' in customers_clean.columns:
    customers_clean['age'] = pd.to_numeric(customers_clean['age'], errors='coerce')
    age_median = customers_clean['age'].median()
    customers_clean['age'] = customers_clean['age'].fillna(age_median)
    print('age 중앙값:', age_median)

if 'city' in customers_clean.columns:
    customers_clean['city'] = customers_clean['city'].fillna('Unknown')

missing_summary(customers_clean)


age 중앙값: 40.0


,missing_count,missing_ratio
customer_id,0,0.0
name,0,0.0
gender,0,0.0
age,0,0.0
city,0,0.0
signup_date,0,0.0


##### 9. 중복 확인하기

중복은 맥락에 따라 의미가 다릅니다. `customers`의 `customer_id` 중복은 오류일 가능성이 크지만, `order_items`의 `order_id` 반복은 한 주문에 여러 상품이 들어갈 수 있으므로 자연스럽습니다.


In [116]:
for name, df in clean_data.items():
    print(name, '전체 행 중복 수:', df.duplicated().sum())


customers 전체 행 중복 수: 0
products 전체 행 중복 수: 0
orders 전체 행 중복 수: 0
order_items 전체 행 중복 수: 0


In [117]:
key_checks = {
    'customers': ('customer_id', customers_clean),
    'products': ('product_id', products_clean),
    'orders': ('order_id', orders_clean),
    'order_items': ('order_item_id', order_items_clean),
}

for name, (key_col, df) in key_checks.items():
    if key_col in df.columns:
        print(name, key_col, '중복 수:', df[key_col].duplicated().sum())
    else:
        print(name, key_col, '컬럼 없음')


customers customer_id 중복 수: 0
products product_id 중복 수: 0
orders order_id 중복 수: 0
order_items order_item_id 중복 수: 0


In [118]:
#원본 type 확인
for name, df in raw_data.items():
    print(f'[{name}]')
    print(df.dtypes.to_dict())

[customers]
{'customer_id': dtype('int64'), 'name': <StringDtype(storage='python', na_value=nan)>, 'gender': <StringDtype(storage='python', na_value=nan)>, 'age': dtype('int64'), 'city': <StringDtype(storage='python', na_value=nan)>, 'signup_date': <StringDtype(storage='python', na_value=nan)>}
[products]
{'product_id': dtype('int64'), 'product_name': <StringDtype(storage='python', na_value=nan)>, 'category': <StringDtype(storage='python', na_value=nan)>, 'price': dtype('int64')}
[orders]
{'order_id': dtype('int64'), 'customer_id': dtype('int64'), 'order_date': <StringDtype(storage='python', na_value=nan)>, 'payment_method': <StringDtype(storage='python', na_value=nan)>, 'order_status': <StringDtype(storage='python', na_value=nan)>}
[order_items]
{'order_item_id': dtype('int64'), 'order_id': dtype('int64'), 'product_id': dtype('int64'), 'quantity': dtype('int64'), 'unit_price': dtype('int64')}


## 1. 전처리 전 상태 기록
- 각 데이터 shape: customers는 150행 6열, products는 100행 4열, orders는 300행 5열, order_items는 764행 5열 이다.
- 결측치: 4개 csv에서 모두 0건이다.
- 전체 중복: 4개 csv 건에서 모두 0건이다. 
- 주요 ID 중복: customer_id, product_id, order_id, order_item_id의 중복 모두 0건이다. 
- 타입 문제 후보: customers.signup_date, orders.order_date.

![전처리 전 상태](images/step01_before.png)

### 결과 관찰
customers는 150행 6열, products는 100행 4열, orders는 300행 5열, order_items는 764행 5열이다. 4개 csv 전체 결측치, 완전 중복 행, id의 중복 모두 0건이다. customers.signup_date와 orders.order_date는 문자열 dtype이므로 날짜 분석 전 변환이 필요한 후보였으며, 가격, 수량, 단가는 정수 dtype이었다.

### 나의 해석과 판단
현재 원본 csv에서는 결측값과 중복 문제가 확인되지 않았으므로 행 삭제나 결측값 대체를 먼저 적용할 필요는 없다고 생각했다. 주요 ID의 중복은 0건이었다. order_items에서 order_id가 반복되는 것은 하나의 주문에 여러 상품이 포함되는 관계에서 자연스럽게 발생할 수 있으므로 구분해서 봐야 한다.

가장 먼저 처리해야 하는 것은 형식 변환이다. 이 외에는 현재 검토한 전처리 상태 중 수정해야 할 부분이 보이지 않기 때문이다. 주문 월, 요일 분석에 필요한 orders.order_date와 고객 가입일 분석에 필요한 customers.signup_date는 문자열 형식이다. 두 컬럼은 먼저 날짜형으로 변환해야 하고, 변환하는  과정에서 변환에 실패하는 값이 있는지 확인해야 한다.

### 업무·분석적 의미
전처리 전 기준선을 정해두면 이후 행 수, 결측값, 중복, 범주값의 변화가 의도한 처리 결과인지 확인할 수 있다. 특히 날짜를 정상적으로 변환해야 기간(ex. 월, 요일)별 주문 분석에서도 계산 범위를 정확하게 잡을 수 있다. 따라서 데이터를 본격적으로 분석하기 전 전처리가 필요한 항목들을 미리 확인해두는 것이 시간 절약 및 데이터 신뢰성에 도움이 된다. 

### 한계와 추가 확인 사항
이번 샘플에서는 품질 문제로 확인된 항목이 모두 0건이었다. 따라서 실제 오류를 처리했을 때 어떤 변화가 생기는지는 확인하지 못했다. 공백만 있는 문자열이나 잘못된 날짜, 중복 ID, 업무상 상태 코드가 실제 운영 데이터에도 없는지는 현재 데이터만으로 판단하기 어렵다. 이 부분은 원본 시스템과 데이터 수집 규칙을 추가로 확인할 필요가 있다. 또한 아직 문자열 변환을 진행하지 못했기 때문에 이를 진행하고 문제가 없는지 확인해야 한다. (추후 step에서 진)

In [119]:
customers_clean = customers_clean.drop_duplicates()
products_clean = products_clean.drop_duplicates()
orders_clean = orders_clean.drop_duplicates()
order_items_clean = order_items_clean.drop_duplicates()

print('완전 중복 행 제거 완료')


완전 중복 행 제거 완료


##### 10. 문자열 표기 정리하기

문자열 데이터에는 앞뒤 공백이 숨어 있을 수 있습니다. `Seoul`과 ` Seoul `은 눈으로는 비슷하지만 pandas에서는 서로 다른 값입니다.

결측치를 문자열 `nan`으로 바꾸지 않도록 주의하면서 문자열 컬럼의 앞뒤 공백을 제거합니다.


In [120]:
def strip_string_columns(df):
    result = df.copy()
    string_columns = result.select_dtypes(include='object').columns

    for col in string_columns:
        result[col] = result[col].where(
            result[col].isna(),
            result[col].astype(str).str.strip()
        )

    return result

customers_clean = strip_string_columns(customers_clean)
products_clean = strip_string_columns(products_clean)
orders_clean = strip_string_columns(orders_clean)
order_items_clean = strip_string_columns(order_items_clean)

print('문자열 앞뒤 공백 제거 완료')


문자열 앞뒤 공백 제거 완료


C:\Users\Song\AppData\Local\Temp\ipykernel_16872\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = result.select_dtypes(include='object').columns
C:\Users\Song\AppData\Local\Temp\ipykernel_16872\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user

In [121]:
if 'city' in customers_clean.columns:
    display(customers_clean['city'].value_counts().head(10))

if 'order_status' in orders_clean.columns:
    display(orders_clean['order_status'].value_counts())


city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64

order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

##### 11. 주문 상태값 통일하기

주문 상태값이 `complete`, `Complete`, `COMPLETED`, `완료`처럼 섞여 있으면 같은 상태를 서로 다른 값으로 집계할 수 있습니다. 대표 표기로 통일합니다.


In [122]:
status_map = {
    'complete': 'completed',
    'Complete': 'completed',
    'COMPLETED': 'completed',
    '완료': 'completed',
    'cancel': 'cancelled',
    'Cancel': 'cancelled',
    'CANCELLED': 'cancelled',
    '취소': 'cancelled',
    'refund': 'refunded',
    'Refund': 'refunded',
    'REFUNDED': 'refunded',
    '환불': 'refunded',
}

if 'order_status' in orders_clean.columns:
    orders_clean['order_status'] = orders_clean['order_status'].replace(status_map)
    display(orders_clean['order_status'].value_counts())
    allowed_statuses = {'completed', 'cancelled', 'refunded'}
    unexpected_statuses = orders_clean.loc[
        ~orders_clean['order_status'].isin(allowed_statuses),
        'order_status',
    ].value_counts(dropna=False)
    print('허용값 밖 order_status:')
    display(unexpected_statuses)


order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

허용값 밖 order_status:


Series([], Name: count, dtype: int64)

##### 12. 날짜 컬럼 변환하기

날짜처럼 보이는 값도 실제로는 문자열일 수 있습니다. 월별 주문 분석이나 요일별 분석을 하려면 날짜형으로 변환해야 합니다.

`errors="coerce"`는 변환할 수 없는 값을 `NaT`로 바꿉니다. 그래서 변환 후 실패 건수를 반드시 확인해야 합니다.


In [123]:
orders_clean['order_date'] = pd.to_datetime(orders_clean['order_date'], errors='coerce')
print('order_date 변환 실패:', orders_clean['order_date'].isna().sum())
print('주문 시작일:', orders_clean['order_date'].min())
print('주문 종료일:', orders_clean['order_date'].max())

orders_clean['order_month'] = orders_clean['order_date'].dt.to_period('M').astype(str)
orders_clean['order_dayofweek'] = orders_clean['order_date'].dt.day_name()

orders_clean[['order_date', 'order_month', 'order_dayofweek']].head()


order_date 변환 실패: 0
주문 시작일: 2025-07-09 00:00:00
주문 종료일: 2026-07-08 00:00:00


,order_date,order_month,order_dayofweek
0,2026-05-07,2026-05,Thursday
1,2025-07-23,2025-07,Wednesday
2,2025-11-19,2025-11,Wednesday
3,2026-01-30,2026-01,Friday
4,2025-12-21,2025-12,Sunday


In [124]:
if 'signup_date' in customers_clean.columns:
    customers_clean['signup_date'] = pd.to_datetime(customers_clean['signup_date'], errors='coerce')
    print('signup_date 변환 실패:', customers_clean['signup_date'].isna().sum())
    display(customers_clean[['customer_id', 'signup_date']].head())


signup_date 변환 실패: 0


,customer_id,signup_date
0,1,2024-06-19
1,2,2025-11-02
2,3,2024-06-12
3,4,2026-04-13
4,5,2024-09-13


##### 13. 숫자형 컬럼 변환하기

가격, 수량, 단가는 숫자처럼 보여도 문자열일 수 있습니다. 특히 `10,000`처럼 쉼표가 들어간 값은 바로 계산하기 어렵습니다.


In [125]:
def to_number(series):
    return pd.to_numeric(
        series.astype(str).str.replace(',', '', regex=False),
        errors='coerce'
    )

products_clean['price'] = to_number(products_clean['price'])
order_items_clean['quantity'] = to_number(order_items_clean['quantity'])
order_items_clean['unit_price'] = to_number(order_items_clean['unit_price'])

print('price 변환 실패:', products_clean['price'].isna().sum())
print('quantity 변환 실패:', order_items_clean['quantity'].isna().sum())
print('unit_price 변환 실패:', order_items_clean['unit_price'].isna().sum())


price 변환 실패: 0
quantity 변환 실패: 0
unit_price 변환 실패: 0


In [126]:
# 변환한 뒤 결측 처리와 처리 근거 확인

for df, col in [
    (customers_clean, 'age'),
    (products_clean, 'price'),
    (order_items_clean, 'unit_price'),
    (order_items_clean, 'quantity')
]:
    missing_before = df[col].isna().sum()

    if df[col].notna().any():
        median = df[col].median()
        df[col] = df[col].fillna(median)
    else:
        median = pd.NA

    print(f'{col} 대체 전 결측:', missing_before)
    print(f'{col} 중앙값:', median)
    print(f'{col} 대체 후 결측:', df[col].isna().sum())

city_missing_before = customers_clean['city'].isna().sum()
customers_clean['city'] = customers_clean['city'].fillna('Unknown')
gender_missing_before = customers_clean['gender'].isna().sum()
customers_clean['gender'] = customers_clean['gender'].fillna('Unknown')

print('city 대체 전 결측:', city_missing_before)
print('city 대체 후 결측:', customers_clean['city'].isna().sum())
print('gender 대체 전 결측:', gender_missing_before)
print('gender 대체 후 결측:', customers_clean['gender'].isna().sum())
print(missing_summary(customers_clean))



age 대체 전 결측: 0
age 중앙값: 40.0
age 대체 후 결측: 0
price 대체 전 결측: 0
price 중앙값: 112000.0
price 대체 후 결측: 0
unit_price 대체 전 결측: 0
unit_price 중앙값: 111000.0
unit_price 대체 후 결측: 0
quantity 대체 전 결측: 0
quantity 중앙값: 3.0
quantity 대체 후 결측: 0
city 대체 전 결측: 0
city 대체 후 결측: 0
gender 대체 전 결측: 0
gender 대체 후 결측: 0
             missing_count  missing_ratio
customer_id              0            0.0
name                     0            0.0
gender                   0            0.0
age                      0            0.0
city                     0            0.0
signup_date              0            0.0


In [127]:
#문자열 변경, 변환 실패 확인
def audit_numeric(series, remove_comma=False):
    values = series.astype(str)
    if remove_comma:
        values = values.str.replace(',', '', regex=False)
    converted = pd.to_numeric(values, errors='coerce')
    return int((~series.isna() & converted.isna()).sum())

def audit_date(series):
    converted = pd.to_datetime(series, errors='coerce')
    return int((~series.isna() & converted.isna()).sum())

for name, df in raw_data.items():
    for col in df.select_dtypes(include='object').columns:
        present = df.loc[df[col].notna(), col].astype(str)
        stripped = present.str.strip()
        print(
            f'{name}.{col} 공백 제거 변경:',
            int((present != stripped).sum()),
            '/ 빈 문자열→결측:',
            int((stripped == '').sum()),
        )

print('age 새 변환 실패:', audit_numeric(customers['age']))
print('price 새 변환 실패:', audit_numeric(products['price'], remove_comma=True))
print('quantity 새 변환 실패:', audit_numeric(order_items['quantity'], remove_comma=True))
print('unit_price 새 변환 실패:', audit_numeric(order_items['unit_price'], remove_comma=True))
print('order_date 새 변환 실패:', audit_date(orders['order_date']))
print('signup_date 새 변환 실패:', audit_date(customers['signup_date']))


customers.name 공백 제거 변경: 0 / 빈 문자열→결측: 0
customers.gender 공백 제거 변경: 0 / 빈 문자열→결측: 0
customers.city 공백 제거 변경: 0 / 빈 문자열→결측: 0
customers.signup_date 공백 제거 변경: 0 / 빈 문자열→결측: 0
products.product_name 공백 제거 변경: 0 / 빈 문자열→결측: 0
products.category 공백 제거 변경: 0 / 빈 문자열→결측: 0
orders.order_date 공백 제거 변경: 0 / 빈 문자열→결측: 0
orders.payment_method 공백 제거 변경: 0 / 빈 문자열→결측: 0
orders.order_status 공백 제거 변경: 0 / 빈 문자열→결측: 0
age 새 변환 실패: 0
price 새 변환 실패: 0
quantity 새 변환 실패: 0
unit_price 새 변환 실패: 0
order_date 새 변환 실패: 0
signup_date 새 변환 실패: 0


C:\Users\Song\AppData\Local\Temp\ipykernel_16872\4198847542.py:14: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:
C:\Users\Song\AppData\Local\Temp\ipykernel_16872\4198847542.py:14: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/

## 2. 문자열·타입·결측 처리
- 적용한 문자열 정리: 앞 뒤 공백 제거, 주문 상태값 통일, 빈 문자열 결측 변환, 숫자형 변환(중간에 , 들어간 경우 제거)
- 숫자 변환 대상과 실패 건수: age, price, quantity, unit_price
- 날짜 변환 대상과 실패 건수: order_date, signup_date
- 결측 처리 기준:
 - 숫자형: 중앙값으로 처리  (Ex. age, price, quantity, unit_price 등) 
 - 범주형: Unknown 으로 처리  (ex. city, gender)

![타입과 결측 처리](images/step02_cleaning.png)

### 결과 관찰
문자열 컬럼의 앞뒤 공백을 제거하고 빈 문자열을 결측값으로 변환한 결과, customers.name, customers.gender, customers.city, customers.signup_date, products.product_name, products.category, orders.order_date, orders.payment_method, orders.order_status에서 변경된 값은 모두 0건이었다. age, price, quantity, unit_price를 숫자형으로 변환했을 때 새롭게 발생한 변환 실패도 모두 0건이었다. order_date와 signup_date의 날짜 변환 실패 역시 각각 0건이었다. order_date의 유효 범위는 2025-07-09부터 2026-07-08까지였다. age, price, quantity, unit_price는 대체 전부터 결측값이 0건이었다. city, gender 역시 결측값이 없어 Unknown 처리 전후 모두 0건으로 유지되었다.

### 나의 해석과 판단
왜 해당 처리 방식을 선택했는지 작성하세요. 삭제/대체가 정보 손실을 만들 가능성도 설명하세요.

문자열은 앞뒤 공백을 제거하고, 빈 문자열은 결측값으로 변환하도록 처리했다. 이번 데이터에서는 실제로 변경된 값이 0건이었다. 다만 범주가 불필요하게 나뉘거나 숨은 결측값이 생기는 것을 막기 위해 이 규칙은 유지했다. 숫자와 날짜는 기존 결측값과 변환 실패를 구분해서 확인했다. 변환 과정에서 새롭게 발생한 실패는 없었다.
age, price, quantity, unit_price의 경우 결측값이 있으면 중앙값으로 대체하도록 했다. 결측치라고 완전히 삭제해버리는 경우 결측치가 많아지면 데이터 분석에 지장이 갈 수 있으며, 그렇다고 랜덤값으로 대체하기에는 범위가 너무 넓어질 가능성이 높기 때문이다. 게다가 숫자형이므로 중앙값을 확인할 수 있기 때문에 이를 선택했다. 반면 city, gender 등의 범주형 값은 결측치가 있으면 Unknown으로 구분하도록 했다. 범주이기 때문에 무작위로 선택하기에는 변동성이 커지고, 직접 확인해야하기 떄문이다. 다만 이번 데이터에서는 모두 실제 대체 대상은 없었다.

### 업무·분석적 의미
문자열의 표기 차이나 변환 실패를 확인하지 않으면 범주별 집계, 금액 계산, 기간 분석에서 오류가 생길 수 있다. 따라서 데이터 분석 전 미리 처리하는 것이 중요하다. 이번 데이터에서는 실제로 변경된 값이 없었다. 다만 결측값과 변환 실패에 대한 처리 규칙을 미리 정해두었다. 이후 같은 구조의 데이터가 들어와도 동일한 기준으로 처리할 수 있도록 하기 위해서다.

### 한계와 추가 확인 사항
현재 데이터 상에는 공백 문자열, 숫자 변환 실패, 날짜 변환 실패가 없었다. 주요한 데이터값의 결측값도 없어서 실제 대체가 데이터 분포에 어떤 영향을 주는지는 확인하지 못했다. 실제 업무에 적용하기 전에는 결측값이 단순 입력 누락인지 수집 과정의 오류인지 먼저 확인할 필요가 있으므로 이를 확인해야 한다. 

##### 14. 이상값 후보 확인하기

이상값은 일반적인 범위를 벗어난 값입니다. 하지만 이상값이 항상 오류는 아닙니다. 0원 상품은 이벤트 상품일 수 있고, 음수 수량은 반품을 의미할 수도 있습니다. 먼저 확인하고 기준을 정해야 합니다.


In [128]:
display(products_clean['price'].describe())
display(order_items_clean[['quantity', 'unit_price']].describe())


count       100.000000
mean     110040.000000
std       56433.910574
min        5000.000000
25%       65750.000000
50%      112000.000000
75%      161000.000000
max      200000.000000
Name: price, dtype: float64

,quantity,unit_price
count,764.000000,764.000000
mean,3.053665,108561.518325
std,1.410873,56996.770604
min,1.000000,5000.000000
25%,2.000000,62000.000000
50%,3.000000,111000.000000
75%,4.000000,161250.000000
max,5.000000,200000.000000


In [129]:
print('price <= 0:', len(products_clean[products_clean['price'] <= 0]))
print('quantity <= 0:', len(order_items_clean[order_items_clean['quantity'] <= 0]))
print('unit_price <= 0:', len(order_items_clean[order_items_clean['unit_price'] <= 0]))


price <= 0: 0
quantity <= 0: 0
unit_price <= 0: 0


In [130]:
print(
    'age < 0 or age > 120:',
    len(customers_clean[(customers_clean['age'] < 0) | (customers_clean['age'] > 120)]),
)

age < 0 or age > 120: 0


###### 실습용 이상값 처리 기준

이번 실습에서는 정상 주문 분석을 위해 다음 기준을 적용합니다.

```text
- age가 0 미만 또는 120 초과
- price가 0 이하인 상품은 분석 대상에서 제외
- quantity가 0 이하인 주문 상세는 분석 대상에서 제외
- unit_price가 0 이하인 주문 상세는 분석 대상에서 제외
```

실제 업무에서는 이 기준을 적용하기 전에 반드시 원본 시스템이나 담당자 확인이 필요합니다.


In [131]:
products_clean = products_clean[products_clean['price'] > 0]
order_items_clean = order_items_clean[order_items_clean['quantity'] > 0]
order_items_clean = order_items_clean[order_items_clean['unit_price'] > 0]

print('이상값 처리 후 products:', products_clean.shape)
print('이상값 처리 후 order_items:', order_items_clean.shape)


이상값 처리 후 products: (100, 4)
이상값 처리 후 order_items: (764, 5)


In [132]:
outlier_candidate_counts = {
    'age < 0 or age > 120': len(
        customers_clean[(customers_clean['age'] < 0) | (customers_clean['age'] > 120)]
    ),
    'price <= 0': len(products_clean[products_clean['price'] <= 0]),
    'quantity <= 0': len(order_items_clean[order_items_clean['quantity'] <= 0]),
    'unit_price <= 0': len(order_items_clean[order_items_clean['unit_price'] <= 0]),
}

print('이상값 후보 보류(자동 제외 없음):', outlier_candidate_counts)


이상값 후보 보류(자동 제외 없음): {'age < 0 or age > 120': 0, 'price <= 0': 0, 'quantity <= 0': 0, 'unit_price <= 0': 0}


In [133]:
for col in ['gender', 'city']:
    print(col, customers_clean[col].unique())

for col in ['category']:
    print(col, products_clean[col].unique())

for col in ['payment_method', 'order_status']:
    print(col, orders_clean[col].unique())

gender <StringArray>
['F', 'M']
Length: 2, dtype: str
city <StringArray>
['광주', '대구', '성남', '울산', '부산', '인천', '수원', '서울', '대전', '고양']
Length: 10, dtype: str
category <StringArray>
['전자기기', '도서', '생활용품', '식품', '스포츠', '패션', '뷰티']
Length: 7, dtype: str
payment_method <StringArray>
['card', 'naver_pay', 'bank_transfer', 'kakao_pay']
Length: 4, dtype: str
order_status <StringArray>
['completed', 'cancelled', 'refunded']
Length: 3, dtype: str


In [134]:
maps = {
    'gender': {'m':'male', '남':'male', '남성':'male', 'f':'female', '여':'female', '여성':'female'},
    'payment_method': {'card':'card', '신용카드':'card', '카드':'card', '계좌이체':'transfer'},
    'order_status': {'complete':'completed', '완료':'completed', 'cancel':'cancelled', '취소':'cancelled', '환불':'refunded'}
}

for df in [customers_clean, products_clean, orders_clean]:
    for col, mapping in maps.items():
        if col in df:
            df[col] = df[col].str.strip().str.lower().replace(mapping)

In [135]:
allowed = {
    'gender': ['female', 'male'],
    'payment_method': ['card', 'naver_pay', 'bank_transfer', 'kakao_pay'],
    'order_status': ['completed', 'cancelled', 'refunded']
}

for df in [customers_clean, products_clean, orders_clean]:
    for col, ok in allowed.items():
        if col in df:
            other = df.loc[~df[col].isin(ok), col].dropna().unique()
            if len(other):
                print(f'{col}의 다른 값:', other)
            else:
                print(f'{col}: 허용값 밖 범주 없음')

gender: 허용값 밖 범주 없음
payment_method: 허용값 밖 범주 없음
order_status: 허용값 밖 범주 없음


## 3. 중복·범주 표준화·이상값 후보
- 제거/보류한 중복: 없음 
- 표준화한 범주값: gender, payment_method, order_status
- 허용값 밖 값: 없음
- 이상값 후보: 없음

![중복 범주 이상값](images/step03_rules.png)

### 결과 관찰
완전 중복 행은 customers, products, orders, order_items에서 모두 0건이었다. 따라서 중복 제거로 삭제된 행도 없었다. order_status, gender, payment_method, order_status는 표준화 전후 값이 같았다. completed 184건, cancelled 64건, refunded 52건이었고, 허용된 값 밖의 상태는 0건이었다. 이상값 후보도 확인했다. age가 0 미만이거나 120을 초과한 경우, price가 0 이하인 경우, quantity가 0 이하인 경우, unit_price가 0 이하인 경우는 모두 0건이었다. 이상값 후보는 바로 제외하지 않고 보류하도록 했다. 이번 데이터에서는 후보 자체가 0건이어서 보류나 제외로 인한 행 수 변화도 없었다.

### 나의 해석과 판단
완전 중복 행은 없어서 제거할 대상도 없었다. 주문 상태도 이미 허용된 세 가지 값으로 구성되어 있었다. 따라서 매핑 규칙은 유지했지만 실제로 변경된 값은 없었다.
이상값 후보 역시 0건이어서 이번 실습해서는 삭제할 건이 없었다. 다만, 이상값을 바로 자동으로 삭제하는 것이 아니라 이상값 후보로 두고, 후에 검토할 수 있도록 하였다. 그 이유는 이상값이 있어도 그것이 0원 상품, 음수 수량, 0원 단가는 증정품이나 반품, 프로모션 같은 정상적인 업무 상황에서도 생길 수 있는 값일 가능성이 있기 때문이다. 그러므로이런 값은 바로 오류로 판단하거나 삭제하지 않고, 우선 확인이 필요한 후보로 두어야 한다.

### 업무·분석적 의미
중복, 범주값, 이상값을 기준 없이 자동 처리하면 정상적인 주문 상세나 반품, 프로모션 기록까지 제거할 수 있다. 상태별 집계 결과도 달라질 수 있다. 따라서 바로 삭제하는 것이 아니라, 허용값과 이상값 후보 기준을 미리 정해두면 이후 같은 문제가 발생했을 때 왜 값이 변경됐는지, 어느 범위까지 영향을 받았는지 확인할 수 있다. 

### 한계와 추가 확인 사항
이번 샘플에서는 중복, 비표준 주문 상태, 이상값 후보가 모두 0건이었다. 따라서 실제로 어떤 값을 보류하거나 제외해야 하는지는 확인하지 못했다. 실제 운영 데이터에 적용할 때는 주문 상태 코드의 정의와 반품 기록 방식, 증정품이나 프로모션의 가격 처리 기준을 먼저 확인해야 한다. 그 기준을 확인한 뒤 보류나 제외 여부를 결정하는 것이 적절하다.


##### 15. 파생 컬럼 만들기

전처리된 수량과 단가를 사용하면 주문 상세 금액을 계산할 수 있습니다.

`line_total = quantity × unit_price`


In [136]:
order_items_clean['line_total'] = (
    order_items_clean['quantity'] * order_items_clean['unit_price']
)

order_items_clean[['quantity', 'unit_price', 'line_total']].head()


,quantity,unit_price,line_total
0,3,102000,306000
1,5,25000,125000
2,3,142000,426000
3,3,193000,579000
4,4,189000,756000


In [137]:
print('전처리 후 주문 상세 금액 합계:', order_items_clean['line_total'].sum())


전처리 후 주문 상세 금액 합계: 255610000


##### 16. 파일 간 관계 다시 확인하기

전처리 과정에서 일부 행을 삭제하면 파일 간 관계가 깨질 수 있습니다. 예를 들어 `products`에서 일부 상품을 제외했는데 `order_items`에는 그 상품 ID가 남아 있을 수 있습니다.


In [138]:
invalid_customers = orders_clean[
    ~orders_clean['customer_id'].isin(customers_clean['customer_id'])
]

invalid_orders = order_items_clean[
    ~order_items_clean['order_id'].isin(orders_clean['order_id'])
]

invalid_products = order_items_clean[
    ~order_items_clean['product_id'].isin(products_clean['product_id'])
]

relationship_checks = pd.DataFrame({
    'check': [
        'orders.customer_id exists in customers.customer_id',
        'order_items.order_id exists in orders.order_id',
        'order_items.product_id exists in products.product_id',
    ],
    'invalid_count': [
        len(invalid_customers),
        len(invalid_orders),
        len(invalid_products),
    ],
})

relationship_checks


,check,invalid_count
0,orders.customer_id exists in customers.custome...,0
1,order_items.order_id exists in orders.order_id,0
2,order_items.product_id exists in products.prod...,0


## 4. PK/FK와 파생 컬럼 재검증
- 없는 `customer_id`: 0건
- 없는 `order_id`: 0건
- 없는 `product_id`: 0건 
- 생성한 파생 컬럼: line_total
- `line_total` 검증: 모두 수량과 단가의 곱이 `line_total`과 모두 일치했으며, 전체 합계는 255,610,000원이었음, invalid_count=0이었음.

![관계와 파생 컬럼 검증](images/step04_validation.png)

### 결과 관찰
orders.customer_id가 customers.customer_id에 없는 경우, order_items.order_id가 orders.order_id에 없는 경우, order_items.product_id가 products.product_id에 없는 경우 뫃두 0건이었다. order_items에는 line_total을 생성했는데, quantity와 unit_price의 곱으로 계산했다. 표본을 확인한 결과 3 × 102000은 306000, 5 × 25000은 125000으로 계산되어 산식과 일치했다. 전처리 후 주문 상세 금액 합계는 255610000이었다. invalid_count 역시 0이었다.

### 나의 해석과 판단
전처리 후 관계를 다시 검증해야 하는 이유를 작성하세요.
3개 PK/FK 관계에서 유효하지 않은 값이 모두 0건이었다. 따라서 이번 전처리 과정에서 고객, 주문, 상품 사이의 참조 관계가 깨지지 않았다고 판단할 수 있었다. 또한 line_total 역시 직접 확인하여 계산이 맞는지 검증했다.
이처럼 전처리 후 관계를 다시 검증해야 하는 이유는 전처리 과정에서 중복 행이나 이상값 행을 제거하면 주문과 주문상세처럼 서로 연결된 데이터의 관계가 깨질 수 있기 때문이다. 예를 들어 고객이나 상품 행이 삭제됐는데 주문이나 주문상세에는 해당 ID가 그대로 남아 있으면 이후 병합이나 집계에서 누락이나 오류가 생길 수 있다. 그래서 전처리가 끝난 뒤에도 각 FK가 실제 PK를 참조하고 있는지 다시 확인해야 한다. 이 검증이 있어야 이후 분석 결과도 신뢰할 수 있다고 생각한다. 

##### 17. 전처리 전후 비교하기

전처리 전후 데이터 크기를 비교합니다. 행 수가 줄었다면 어떤 기준으로 줄었는지 설명할 수 있어야 하고, 열 수가 늘었다면 어떤 파생 컬럼이 추가되었는지 설명할 수 있어야 합니다.


In [139]:
processed_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

processed_shapes = pd.DataFrame({
    'dataset': list(processed_data.keys()),
    'rows': [df.shape[0] for df in processed_data.values()],
    'columns': [df.shape[1] for df in processed_data.values()],
})

comparison = raw_shapes.merge(
    processed_shapes,
    on='dataset',
    suffixes=('_raw', '_processed')
)

comparison


,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,150,6,150,6
1,products,100,4,100,4
2,orders,300,5,300,7
3,order_items,764,5,764,6


In [140]:
#전처리 전후 상세 비교
def new_numeric_failure(series, remove_comma=False):
    values = series.astype(str)
    if remove_comma:
        values = values.str.replace(',', '', regex=False)
    converted = pd.to_numeric(values, errors='coerce')
    return int((~series.isna() & converted.isna()).sum())

def new_date_failure(series):
    converted = pd.to_datetime(series, errors='coerce')
    return int((~series.isna() & converted.isna()).sum())

conversion_failures = {
    'age': new_numeric_failure(customers['age']),
    'price': new_numeric_failure(products['price'], remove_comma=True),
    'quantity': new_numeric_failure(order_items['quantity'], remove_comma=True),
    'unit_price': new_numeric_failure(order_items['unit_price'], remove_comma=True),
    'order_date': new_date_failure(orders['order_date']),
    'signup_date': new_date_failure(customers['signup_date']),
}

q5_comparison = pd.DataFrame([
    {
        'dataset': name,
        'rows_before': raw_data[name].shape[0],
        'rows_after': processed_data[name].shape[0],
        'missing_before': int(raw_data[name].isna().sum().sum()),
        'missing_after': int(processed_data[name].isna().sum().sum()),
        'duplicate_before': int(raw_data[name].duplicated().sum()),
        'duplicate_after': int(processed_data[name].duplicated().sum()),
    }
    for name in raw_data
])

print(q5_comparison.to_string(index=False))
print('새 변환 실패:', conversion_failures)

    dataset  rows_before  rows_after  missing_before  missing_after  duplicate_before  duplicate_after
  customers          150         150               0              0                 0                0
   products          100         100               0              0                 0                0
     orders          300         300               0              0                 0                0
order_items          764         764               0              0                 0                0
새 변환 실패: {'age': 0, 'price': 0, 'quantity': 0, 'unit_price': 0, 'order_date': 0, 'signup_date': 0}


In [141]:
before = {
    'customers': customers, 'products': products,
    'orders': orders, 'order_items': order_items
}
after = {
    'customers': customers_clean, 'products': products_clean,
    'orders': orders_clean, 'order_items': order_items_clean
}

for name in before:
    print(
        f'{name}: 행 {len(before[name])} → {len(after[name])}, '
        f'열 {before[name].shape[1]} → {after[name].shape[1]}, '
        f'결측 {before[name].isna().sum().sum()} → {after[name].isna().sum().sum()}, '
        f'완전 중복 {before[name].duplicated().sum()} → {after[name].duplicated().sum()}'
    )

def numeric_fail(s):
    return pd.to_numeric(
        s.dropna().astype(str).str.replace(',', '', regex=False),
        errors='coerce'
    ).isna().sum()

def date_fail(s):
    return pd.to_datetime(s.dropna(), errors='coerce').isna().sum()

print('새 변환 실패:', {
    'age': numeric_fail(customers['age']),
    'price': numeric_fail(products['price']),
    'quantity': numeric_fail(order_items['quantity']),
    'unit_price': numeric_fail(order_items['unit_price']),
    'order_date': date_fail(orders['order_date']),
    'signup_date': date_fail(customers['signup_date'])
})

customers: 행 150 → 150, 열 6 → 6, 결측 0 → 0, 완전 중복 0 → 0
products: 행 100 → 100, 열 4 → 4, 결측 0 → 0, 완전 중복 0 → 0
orders: 행 300 → 300, 열 5 → 7, 결측 0 → 0, 완전 중복 0 → 0
order_items: 행 764 → 764, 열 5 → 6, 결측 0 → 0, 완전 중복 0 → 0
새 변환 실패: {'age': np.int64(0), 'price': np.int64(0), 'quantity': np.int64(0), 'unit_price': np.int64(0), 'order_date': np.int64(0), 'signup_date': np.int64(0)}


## 5. 전처리 전/후 비교
| 항목 | 처리 전 | 처리 후 | 변화 이유 |
| --- | ---: | ---: | --- |
| 행 수 | customers 150, products 100, orders 300, order_items 764 | customers 150, products 100, orders 300, order_items 764 | 변화 X |
| 결측 | 0 | 0 | 변화 X |
| 중복 | 0 | 0 | 변화 X |
| 변환 실패 | - | age, price, quantity, unit_price, order_date, signup_date 모두 실패 0건 | 변화 X |

![전처리 전후 비교](images/step05_before_after.png)

### 결과 관찰
데이터에서 전처리 전후 행수, 결측, 중복 모두 0건으로 변화가 없었으며, 변환 실패 역시 0이었다. 
다만 orders 의 경우 열이 5열에서 7열로 증가했고, orders_items는 6열로 증가했다. 

### 나의 해석과 판단
행 수와 결측값, 중복 수가 그대로였는데, 이는 처리할 후보가 없었고 변환 실패도 발생하지 않았던 결과와 일치한다. 또한 결과가 그러했으므로 행수, 값분포 변화가 없는 결과는 내가 의도한 바와 같다. 
다만 orders와 orders_items에서 열이 증가했는데, 이는 분석에 필요한 파생 컬럼을 추가했기 때문이다. orders는 order_month와 order_dayofweek가 추가었고, order_items는 line_total이 추가되어 5열에서 6열로 늘었다. 따라서 증가한 열이 추가한 열의 수와 같기 때문에 이를 근거로 의도한 결과라고 볼 수 있다. 

### 업무·분석적 의미
전후 비교 결과를 남겨두면 이후 EDA에서 행 수 감소나 분포 변화가 전처리 때문인지 구분할 수 있기 때문에 이를 만들어두는 것은 업무에서 중요하다. 이번에는 주문 월,요일과 주문 상세 금액을 추가했는데, 이를 통해서 기간별, 요일별, 상품 단위별 분석을 추후에 진행할 수 있을 것이다.

### 한계와 추가 확인 사항
이번 샘플에는 결측값, 중복, 변환 실패, 이상값 후보가 없었다. 그래서 전처리로 인한 실제 정보 손실이나 분포 변화까지는 확인하지 못했다. 또한 변환 실패의 처리 전 값은 문제가 없었기 떄문에 아직 변환을 시도하지 않은 상태라 직접 비교하기 어려웠다. 

##### 18. 전처리 결과 저장하기

전처리된 데이터는 `data/processed` 폴더에 저장합니다. Excel에서 한글 CSV를 바로 열 수 있도록 `utf-8-sig` 인코딩을 사용합니다.


In [142]:
customers_clean.to_csv(PROCESSED_DIR / 'customers_clean.csv', index=False, encoding='utf-8-sig')
products_clean.to_csv(PROCESSED_DIR / 'products_clean.csv', index=False, encoding='utf-8-sig')
orders_clean.to_csv(PROCESSED_DIR / 'orders_clean.csv', index=False, encoding='utf-8-sig')
order_items_clean.to_csv(PROCESSED_DIR / 'order_items_clean.csv', index=False, encoding='utf-8-sig')

list(PROCESSED_DIR.glob('*_clean.csv'))


[WindowsPath('c:/dev/llm-data-analysis-course/data/processed/customers_clean.csv'),
 WindowsPath('c:/dev/llm-data-analysis-course/data/processed/orders_clean.csv'),
 WindowsPath('c:/dev/llm-data-analysis-course/data/processed/order_items_clean.csv'),
 WindowsPath('c:/dev/llm-data-analysis-course/data/processed/products_clean.csv')]

##### 19. 전처리 요약 보고서 저장하기

전처리 기준과 결과를 Markdown 파일로 남겨 두면 이후 보고서 작성이 쉬워집니다.


In [143]:
summary_text = f'''# Chapter 5 데이터 전처리 요약

##### 전처리 결과 파일

- customers_clean.csv
- products_clean.csv
- orders_clean.csv
- order_items_clean.csv

##### 전처리 전후 데이터 크기

```text
{comparison.to_string(index=False)}
```

##### 파일 간 관계 점검 결과

```text
{relationship_checks.to_string(index=False)}
```

##### 주요 처리 내용

- 원본 데이터를 직접 수정하지 않고 복사본을 사용함
- 문자열 컬럼 앞뒤 공백 제거
- 고객 나이 결측치는 중앙값으로 대체
- 고객 도시 결측치는 Unknown으로 처리
- 주문 상태값 표기 통일
- 날짜 컬럼 변환 및 주문 월/요일 파생 컬럼 생성
- 가격, 수량, 단가를 숫자형으로 변환
- 0 이하 가격, 수량, 단가 확인 및 처리
- line_total 파생 컬럼 생성
- 파일 간 키 관계 재확인

##### 주의 사항

이 전처리 기준은 실습용 예시입니다. 실제 업무에서는 결측치와 이상값을 삭제하거나 대체하기 전에 원본 시스템, 수집 과정, 업무 담당자 확인이 필요합니다.
'''

report_path = REPORT_DIR / 'ch05_preprocessing_summary.md'
report_path.write_text(summary_text, encoding='utf-8')

print('요약 보고서 저장 완료:', report_path)


요약 보고서 저장 완료: c:\dev\llm-data-analysis-course\reports\ch05_preprocessing_summary.md


In [144]:
#비교용
summary_text = f'''# Chapter 5 데이터 전처리 요약

##### 전처리 결과 파일

- customers_clean.csv
- products_clean.csv
- orders_clean.csv
- order_items_clean.csv

##### 전처리 전후 데이터 크기

```text
{comparison.to_string(index=False)}
```

##### 파일 간 관계 점검 결과

```text
{relationship_checks.to_string(index=False)}
```

##### 주요 처리 내용

- 원본 데이터를 직접 수정하지 않고 복사본을 사용함
- 문자열 컬럼 앞뒤 공백 제거
- 고객 나이 결측치는 중앙값으로 대체
- 고객 도시 결측치는 Unknown으로 처리
- 주문 상태값 표기 통일
- 날짜 컬럼 변환 및 주문 월/요일 파생 컬럼 생성
- 가격, 수량, 단가를 숫자형으로 변환
- 0 이하 가격, 수량, 단가 확인 및 처리
- line_total 파생 컬럼 생성
- 파일 간 키 관계 재확인

##### 주의 사항

이 전처리 기준은 실습용 예시입니다. 실제 업무에서는 결측치와 이상값을 삭제하거나 대체하기 전에 원본 시스템, 수집 과정, 업무 담당자 확인이 필요합니다.
'''

report_path = REPORT_DIR / 'ch05_preprocessing_summary_1.md'
report_path.write_text(summary_text, encoding='utf-8')

print('요약 보고서 저장 완료:', report_path)


요약 보고서 저장 완료: c:\dev\llm-data-analysis-course\reports\ch05_preprocessing_summary_1.md


##### 20. 전처리 과정을 함수로 정리하기

전처리 코드를 한 번만 실행하고 끝내면 재사용하기 어렵습니다. 같은 데이터를 다시 받거나 처리 기준을 조금 바꾸어야 할 때는 함수로 정리된 코드가 훨씬 유용합니다.


In [145]:
def preprocess_customers(df):
    result = strip_string_columns(df)

    if 'age' in result.columns:
        result['age'] = pd.to_numeric(result['age'], errors='coerce')
        result['age'] = result['age'].fillna(result['age'].median())

    if 'city' in result.columns:
        result['city'] = result['city'].fillna('Unknown')

    if 'signup_date' in result.columns:
        result['signup_date'] = pd.to_datetime(result['signup_date'], errors='coerce')

    return result.drop_duplicates()


def preprocess_products(df):
    result = strip_string_columns(df)

    if 'price' in result.columns:
        result['price'] = to_number(result['price'])
        result = result[result['price'] > 0]

    return result.drop_duplicates()


def preprocess_orders(df):
    result = strip_string_columns(df)

    if 'order_status' in result.columns:
        result['order_status'] = result['order_status'].replace(status_map)

    if 'order_date' in result.columns:
        result['order_date'] = pd.to_datetime(result['order_date'], errors='coerce')
        result['order_month'] = result['order_date'].dt.to_period('M').astype(str)
        result['order_dayofweek'] = result['order_date'].dt.day_name()

    return result.drop_duplicates()


def preprocess_order_items(df):
    result = strip_string_columns(df)

    if 'quantity' in result.columns:
        result['quantity'] = to_number(result['quantity'])
        result = result[result['quantity'] > 0]

    if 'unit_price' in result.columns:
        result['unit_price'] = to_number(result['unit_price'])
        result = result[result['unit_price'] > 0]

    if {'quantity', 'unit_price'}.issubset(result.columns):
        result['line_total'] = result['quantity'] * result['unit_price']

    return result.drop_duplicates()


In [146]:
customers_clean_func = preprocess_customers(customers)
products_clean_func = preprocess_products(products)
orders_clean_func = preprocess_orders(orders)
order_items_clean_func = preprocess_order_items(order_items)

print(customers_clean_func.shape)
print(products_clean_func.shape)
print(orders_clean_func.shape)
print(order_items_clean_func.shape)


(150, 6)
(100, 4)
(300, 7)
(764, 6)


C:\Users\Song\AppData\Local\Temp\ipykernel_16872\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = result.select_dtypes(include='object').columns
C:\Users\Song\AppData\Local\Temp\ipykernel_16872\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user

##### 21. 소스 모듈 사용하기

위에서 직접 작성한 전처리 함수는 `src/preprocessing.py`에도 정리되어 있습니다. 반복 작업이나 실제 프로젝트에서는 노트북 안에만 코드를 두기보다 소스 모듈로 분리하는 것이 좋습니다.


In [147]:
from src.data_loader import load_sales_data
from src.preprocessing import (
    build_preprocessing_report,
    compare_shapes,
    duplicate_summary,
    preprocess_sales_data,
    save_processed_data,
    validate_relationships,
)

module_raw_data = load_sales_data(RAW_DIR)
module_processed_data = preprocess_sales_data(module_raw_data)
module_relationship_checks = validate_relationships(module_processed_data)
module_comparison = compare_shapes(module_raw_data, module_processed_data)

module_comparison


,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,150,6,150,6
1,order_items,764,5,764,6
2,orders,300,5,300,7
3,products,100,4,100,4


In [148]:
duplicate_summary(
    module_processed_data,
    key_columns={
        'customers': 'customer_id',
        'products': 'product_id',
        'orders': 'order_id',
        'order_items': 'order_item_id',
    },
)


,dataset,row_duplicate_count,key_column,key_duplicate_count
0,customers,0,customer_id,0
1,products,0,product_id,0
2,orders,0,order_id,0
3,order_items,0,order_item_id,0


In [149]:
module_relationship_checks


,check,invalid_count
0,orders.customer_id exists in customers.custome...,0
1,order_items.order_id exists in orders.order_id,0
2,order_items.product_id exists in products.prod...,0


##### 22. 스크립트로 한 번에 실행하기

노트북에서 한 단계씩 이해한 전처리 과정을 스크립트로도 실행할 수 있습니다. 터미널에서 프로젝트 루트 기준으로 아래 명령을 실행합니다.

```bash
python scripts/preprocess_data.py
```

이 스크립트는 전처리 결과 CSV와 요약 보고서를 자동으로 저장합니다.


## 6. 재현 가능한 전처리 확인
- 생성된 clean CSV: customers_clean.csv, products_clean.csv, orders_clean.csv, order_items_clean.csv
- 생성된 요약 보고서: reports/ch05_preprocessing_summary1.md
  - 4개의 clean CSV가 정상적으로 생성되었다.
  - 전처리 전후 데이터 크기와 파일 간 관계 점검 결과가 기록되었다.
  - customers 150행, products 100행, orders 300행, order_items 764행으로 전처리 전후 행 수는 유지되었다.
  - 파일 간 외래키 관계 점검 결과 invalid_count가 모두 0으로 확인되었다.
- `python scripts/preprocess_data.py` 재실행 결과: reports/ch05_preprocessing_summary.md
  - 동일한 4개의 clean CSV가 다시 생성되었다.
  - customers 150행, products 100행, orders 300행, order_items 764행으로 이전 실행과 동일한 데이터 크기가 확인되었다.
  - orders.customer_id, order_items.order_id, order_items.product_id에 대한 관계 점검에서도 invalid_count가 모두 0으로 나타났다.
  - 추가로 각 데이터셋의 행 중복과 기본키 중복 여부를 확인하는 항목이 포함되었으며, 모든 데이터셋에서 중복 개수가 0으로 확인되었다.
- 재실행 후 달라진 점: 핵심 전처리 결과와 생성된 CSV의 데이터 크기는 이전과 같았으나, 다시 실행한 요약 보고서에는 중복 점검 결과가 추가되었다. 일부 데이터셋의 표시 순서와 보고서의 문장 표현, 제목 형식에는 차이가 있었다. 하지만 실제 전처리 결과에는 영향을 주지 않았다.

![재실행 결과](images/step06_reproducible.png)

### 나의 해석과 판단
Notebook에서 전처리가 한 번 정상적으로 실행됐다는 것은 그때 당시의 실행 환경과 셀 순서에서는 원하는 결과가 나왔다는 의미다. 하지만 셀 실행 순서나 이전에 저장된 변수 상태에 따라 결과가 달라질 수 있다. 한 번 성공했다고 해서 항상 같은 결과가 나온다고 보기는 어렵다.
반면 전처리 과정을 하나의 스크립트로 만들면 정해진 순서대로 다시 실행할 수 있다. 이번에도 python scripts/preprocess_data.py를 다시 실행했을 때 같은 clean CSV가 생성됐다. 데이터 크기와 파일 간 관계 점검 결과도 이전과 같았다. 따라서 이번 전처리 과정은 특정 Notebook의 실행 상태에만 의존하지 않고 다시 실행할 수 있다고 생각했다. 실제 데이터 분석에서도 한 번 정상적으로 실행되는 것보다 같은 입력 데이터와 같은 코드를 사용했을 때 같은 결과를 다시 얻을 수 있는지가 중요하다. 그래서 전처리 과정을 스크립트로 정리해 두는 것이 필요하다고 생각했다.

##### 23. LLM과 함께 전처리 코드를 검토하기

LLM에게 전처리 코드를 요청할 때는 실제 고객명, 이메일, 주문 내역을 그대로 입력하지 않는 것이 좋습니다. 컬럼명, 데이터 타입, 결측치 개수, 중복 여부, 처리 목적처럼 구조화된 정보만 제공하세요.


- 사용한 LLM 프롬프트  (내 실습에서는 결측치가 없었으므로 가상의 상황을 가정)
```text

customers 데이터 품질 요약
- rows: 150
- customer_id missing/duplicate: 0 / 0
- age missing: 1
- city missing: 1

요청:

1. age 결측치 처리 후보 비교
2. 각 방법의 장단점 설명
3. 처리 전후 검증 항목 제안

이후 
다음 전처리 코드가 안전한지 검토해 주세요.

customers["age"] = customers["age"].fillna(customers["age"].mean())
customers = customers.drop_duplicates()
orders["order_date"] = pd.to_datetime(orders["order_date"])
order_items["line_total"] = order_items["quantity"] * order_items["unit_price"]

검토 기준:
- 원본 데이터를 직접 수정하는 문제가 있는지
- 결측치 처리 방식이 적절한지
- 날짜 변환 실패를 확인하는지
- 중복 제거 기준이 충분한지
- line_total 계산 전에 숫자형 변환이 필요한지
- 더 안전한 코드로 어떻게 수정할 수 있는지
```


- LLM 답변 요약
    - age 결측치 1건은 행을 삭제하거나 복잡한 예측 모델을 사용하는 것보다 중앙값으로 대체하는 방법을 권장함
    - 처리 전후에 결측치 개수, 기술통계량, 행 수, 분포 변화를 확인하도록 제안함
    - 원본 데이터를 유지하기 위해 전처리 전에 .copy()를 사용하도록 권장함
    - 중복을 제거할 때는 전체 행이 아니라 customer_id를 기준으로 확인하도록 제안함
    - 날짜 변환 과정에서 오류가 발생하지 않도록 pd.to_datetime(..., errors="coerce")를 사용하도록 권장함
    - line_total 계산 전 quantity와 unit_price를 숫자형으로 변환하고, 변환에 실패한 값이 있는지 확인하도록 제안함
    - 전체적으로 데이터 손실을 줄이고 원본을 보존하면서, 자료형과 처리 전후 결과를 확인하는 방향으로 전처리를 진행하도록 제안함

- 검토: 대체로 다 수용 가능한 제안이라고 생각함. 실제로 대부분 실습에서도 진행한 내용임.


##### 24. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 각 데이터셋의 결측치 비율을 하나의 표로 합쳐 보세요.
2. `customers_clean`에서 나이가 18세 미만이거나 100세 초과인 값이 있는지 확인하세요.
3. `orders_clean`에서 월별 주문 건수를 계산하세요.
4. `order_items_clean`에서 `line_total`이 큰 순서대로 상위 10개를 확인하세요.
5. 전처리 전후 행 수가 줄어든 데이터셋이 있는지 설명해 보세요.
6. LLM에게 결측치 처리 기준을 검토해 달라는 프롬프트를 직접 작성해 보세요.


In [151]:
# 준비
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path(r"C:\dev\llm-data-analysis-course")
RAW_DIR = PROJECT_ROOT / "data" / "raw"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_sales_data
from src.preprocessing import preprocess_sales_data

raw_data = load_sales_data(RAW_DIR)
processed_data = preprocess_sales_data(raw_data)

customers_clean = processed_data['customers']
products_clean = processed_data['products']
orders_clean = processed_data['orders']
order_items_clean = processed_data['order_items']

print('24번 실습과제 전처리 상태 준비 완료')


24번 실습과제 전처리 상태 준비 완료


In [152]:
# 과제 1. 각 데이터셋의 결측치 비율을 하나의 표로 합쳐 보세요.
# hint: missing_summary(df).reset_index()를 활용해 보세요.

missing_ratio_table = pd.concat(
    [
        pd.DataFrame(
            {
                'dataset': name,
                'column': df.columns,
                'missing_count': df.isna().sum().to_numpy(),
                'missing_ratio_pct': (df.isna().mean() * 100).round(2).to_numpy(),
            }
        )
        for name, df in processed_data.items()
    ],
    ignore_index=True,
)

print('검증 대상 컬럼 수:', len(missing_ratio_table))
print('최대 결측 수:', int(missing_ratio_table['missing_count'].max()))
print('최대 결측 비율(%):', float(missing_ratio_table['missing_ratio_pct'].max()))
missing_ratio_table


검증 대상 컬럼 수: 23
최대 결측 수: 0
최대 결측 비율(%): 0.0


,dataset,column,missing_count,missing_ratio_pct
0,customers,customer_id,0,0.0
1,customers,name,0,0.0
2,customers,gender,0,0.0
3,customers,age,0,0.0
4,customers,city,0,0.0
5,customers,signup_date,0,0.0
6,products,product_id,0,0.0
7,products,product_name,0,0.0
8,products,category,0,0.0
9,products,price,0,0.0


In [153]:
# 과제 2. customers_clean에서 나이가 18세 미만이거나 100세 초과인 값이 있는지 확인하세요.
age_outlier_18_100 = customers_clean.loc[
    (customers_clean['age'] < 18) | (customers_clean['age'] > 100)
]

age_condition_verified = (
    (age_outlier_18_100['age'] < 18) | (age_outlier_18_100['age'] > 100)
).all()
print('나이 18세 미만 또는 100세 초과 건수:', len(age_outlier_18_100))
print('추출 행의 조건 일치:', bool(age_condition_verified))
age_outlier_18_100

나이 18세 미만 또는 100세 초과 건수: 0
추출 행의 조건 일치: True


,customer_id,name,gender,age,city,signup_date


In [154]:
# 과제 3. orders_clean에서 월별 주문 건수를 계산하세요.
monthly_order_counts = (
    orders_clean.groupby('order_month')
    .size()
    .rename('order_count')
    .reset_index()
    .sort_values('order_month')
)

print('월 수:', len(monthly_order_counts))
print('월별 주문 건수 합계:', int(monthly_order_counts['order_count'].sum()))
print('orders 행 수와 합계 일치:', int(monthly_order_counts['order_count'].sum()) == len(orders_clean))
monthly_order_counts


월 수: 13
월별 주문 건수 합계: 300
orders 행 수와 합계 일치: True


,order_month,order_count
0,2025-07,15
1,2025-08,24
2,2025-09,23
3,2025-10,39
4,2025-11,23
5,2025-12,26
6,2026-01,29
7,2026-02,29
8,2026-03,21
9,2026-04,31


In [155]:
# 과제 4. order_items_clean에서 line_total이 큰 순서대로 상위 10개를 확인하세요.
top10_line_total = (
    order_items_clean[
        ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']
    ]
    .sort_values('line_total', ascending=False)
    .head(10)
)

line_total_recalculated = top10_line_total['quantity'] * top10_line_total['unit_price']
print('상위 행 수:', len(top10_line_total))
print('내림차순 정렬 검증:', top10_line_total['line_total'].is_monotonic_decreasing)
print('quantity × unit_price 산식 일치:', top10_line_total['line_total'].equals(line_total_recalculated))
print('상위 10개 line_total 합계:', int(top10_line_total['line_total'].sum()))
top10_line_total

상위 행 수: 10
내림차순 정렬 검증: True
quantity × unit_price 산식 일치: True
상위 10개 line_total 합계: 9905000


,order_item_id,order_id,product_id,quantity,unit_price,line_total
22,23,10,99,5,200000,1000000
85,86,39,99,5,200000,1000000
183,184,76,99,5,200000,1000000
493,494,190,70,5,198000,990000
78,79,35,70,5,198000,990000
631,632,247,58,5,197000,985000
604,605,235,58,5,197000,985000
113,114,51,58,5,197000,985000
569,570,221,58,5,197000,985000
45,46,21,58,5,197000,985000


In [156]:
#전처리 전후 행수 비교
row_count_comparison = pd.DataFrame(
    [
        {
            'dataset': name,
            'rows_before': raw_data[name].shape[0],
            'rows_after': processed_data[name].shape[0],
            'row_change': processed_data[name].shape[0] - raw_data[name].shape[0],
        }
        for name in raw_data
    ]
)

print('행 수 감소 데이터셋 수:', int((row_count_comparison['row_change'] < 0).sum()))
print('전후 총 행 수:', int(row_count_comparison['rows_before'].sum()), '->', int(row_count_comparison['rows_after'].sum()))
row_count_comparison


행 수 감소 데이터셋 수: 0
전후 총 행 수: 1314 -> 1314


,dataset,rows_before,rows_after,row_change
0,customers,150,150,0
1,products,100,100,0
2,orders,300,300,0
3,order_items,764,764,0


결론적으로 행수 변화가 없었다.

6. LLM에게 결측치 처리 기준을 검토해 달라는 프롬프트를 직접 작성해 보세요.

다음은 온라인 쇼핑몰 데이터 전처리 품질 요약이다. (ch05_preprocessing_summary.md 제공)

다음 전처리 기준을 검토해라. 

1. age 결측은 중앙값으로 대체하고 city 결측은 Unknown으로 처리하는 기준의 장단점을 설명
2. 이상치를 자동 삭제하지 않고 업무 의미를 먼저 확인해야 하는 이유를 설명(내 데이터에 맞게)
3. 처리 전후에 다시 확인해야 할 결측, 중복, 변환 실패, PK/FK 관계 검증 항목을 제안
-실제 업무 의미를 확인하지 않은 값은 오류라고 단정하지 말 것

##### 25. 정리

이번 장에서는 다음 내용을 실습했습니다.

- 원본 데이터와 전처리 데이터 분리
- 결측치 개수와 비율 확인
- 나이 결측치 중앙값 대체, 도시 결측치 Unknown 처리
- 중복 행과 주요 ID 중복 점검
- 문자열 앞뒤 공백 제거와 상태값 표기 통일
- 날짜형 변환과 주문 월/요일 파생 컬럼 생성
- 숫자형 변환과 변환 실패 건수 확인
- 이상값 후보 확인과 실습용 처리 기준 적용
- 주문 상세 금액 `line_total` 생성
- 파일 간 키 관계 재확인
- 전처리 결과 CSV와 요약 보고서 저장
- 전처리 함수를 `src/preprocessing.py` 소스 모듈로 분리

다음 장에서는 전처리된 데이터를 바탕으로 탐색적 데이터 분석, 즉 EDA를 수행합니다.


## 7. Chapter 05 최종 판단
### 내가 정한 전처리 원칙 3가지
1. 원본 데이터는 그대로 두고 복사본에서 전처리를 진행한다. 처리 전후에는 행·열 수, 결측값, 완전 중복, 키 관계를 같은 기준으로 확인하고 기록한다.
2. 문자열을 먼저 정리한 뒤 숫자와 날짜는 errors="coerce"로 변환한다. 이때 기존 결측값과 변환 과정에서 새로 발생한 실패값을 구분해서 확인한다.
3. 이상값 후보는 바로 오류로 판단하지 않는다.

### 가장 위험하다고 생각한 자동 처리 1가지와 이유
가장 주의해야 할 자동 처리는 이상치를 일과적으로 모두 삭제하는 것이라고 생각한다. 예를 들어 이번 실습의 경우 0 이하의 가격, 수량, 단가를 모두 삭제하는 것이다. 이런 경우 실제 데이터에서는 0원 상품이 증정품이나 프로모션일 수 있고, 음수 수량은 반품을 의미할 수도 있다. 이런 값을 바로 삭제하면 정상적인 업무 기록까지 사라진다. 상품 수요나 반품률, 금액 집계 결과도 달라질 수 있다.
따라서 이런 값은 먼저 이상값 후보로 표시해 두는 것이 필요할 것이다. 이후 상태 코드와 주문 정책을 확인하고, 필요한 경우 담당자에게 확인한 뒤 실제 오류인지 정상적인 업무 기록인지 구분해야 한다.

### 다음 EDA에서 특히 주의할 데이터 특성

orders의 날짜 범위와 주문 상태별 분포를 먼저 확인해야 한다. 취소와 환불 주문을 집계에 포함할지도 기준을 정해두어야 한다.

그리고 line_total 값에 유의해야 한다. line_total은 주문 상세 한 행의 quantity × unit_price로 계산한 값이다. 할인, 세금, 배송비, 환불 등이 반영된 매출이나 순매출이라고 단정할 수는 없다. 따라서 EDA에서는 주문 단위와 주문상세 단위를 구분하고, 상태와 기간, 반품 기준에 따라 어떤 데이터를 집계했는지 명확히 해야 한다.

### 현재 전처리의 한계
이번 샘플에서는 결측값, 완전 중복, 키 중복, 새 변환 실패, 이상값 후보가 모두 0건이었다. 그래서 실제 오류를 처리했을 때 행이 얼마나 줄어드는지, 데이터 분포가 어떻게 달라지는지는 확인하지 못했다. 
또한 특히 숫자형 컬럼의 경우 결측치나 이상치를 처리하기 위해 임의로 median 값을 넣기로 설정하였다. 해당 방법이 실제 데이터 특성에 적절한지 충분히 검증해야 하는데, 현재 단계에서는 그까지 진행하지는 않았다. 

## 최종 제출 체크
- [x] raw 원본을 덮어쓰지 않았습니다.
- [x] 처리 전/후 비교를 남겼습니다.
- [x] 처리 기준과 이유를 자신의 말로 작성했습니다.
- [x] clean CSV와 재실행 결과를 확인했습니다.
- [x] 개인정보/Secret이 없습니다.
- [x] `chapter05/chapter05.ipynb`가 GitHub에서 정상 표시됩니다.
- [x] 최종 Notebook 파일 URL을 제출합니다.